# DỰ ĐOÁN VÀ ĐÁNH GIÁ NGUY CƠ BỆNH TIM MẠCH (HEALTHCARE AND PATIENT RISK ANALYSIS)

### Nghiên cứu Đa Tập Dữ Liệu (Multi-Dataset Study), Đánh Giá Xuyên Miền (Cross-Domain), Trôi Dữ Liệu (Data Drift) và Khung Dự Đoán Tin Cậy (Trustworthy AI)

---

## 1. Giới thiệu & Mục tiêu nghiên cứu

Bệnh tim mạch (Cardiovascular Diseases - CVD) là nguyên nhân gây tử vong hàng đầu trên toàn cầu. Việc áp dụng Học Máy để sàng lọc nguy cơ sớm cho bệnh nhân có ý nghĩa sống còn. Tuy nhiên, trong y tế thực tế, mô hình huấn luyện trên một bệnh viện/quần thể (population) hiếm khi hoạt động hoàn hảo khi triển khai tại một bệnh viện khác do hiện tượng **Trôi phân phối đặc trưng (Covariate Shift)**, **Lệch tỷ lệ nền (Base-rate / Prior Shift)** và **Dịch chuyển khái niệm mục tiêu (Concept Shift)**.

Báo cáo nghiên cứu và mã nguồn này giải quyết trọn vẹn 4 yêu cầu của Bài tập lớn:

1. **Sử dụng từ 2 bộ dữ liệu y tế độc lập**:
   - **Tập A: Framingham Heart Study** (4.240 bệnh nhân, Mỹ, tỷ lệ bệnh ~15%, bài toán dự đoán biến cố tim mạch 10 năm - _TenYearCHD_).
   - **Tập B: Cardiovascular Disease Dataset** (70.000 bệnh nhân, Đông Âu, tỷ lệ bệnh ~50%, bài toán chẩn đoán hiện diện bệnh tim mạch - _cardio_).
2. **Tổng hợp và so sánh toàn diện các mô hình học máy**:
   - Baseline
   - _Logistic Regression_
   - _K-Nearest Neighbors (KNN)_
   - _Gaussian Naive Bayes_
   - _Random Forest_
   - _XGBoost_
   - _Support Vector Machine (SVM)_
3. **Phân tích lỗi chuyên sâu, đánh giá đúng chuẩn y tế (Model Assessment)**:
   - Đơn vị quan sát độc lập là từng bệnh nhân riêng biệt; phân chia 60% Train / 20% Validation / 20% Test phân tầng (_Stratified Split_), tinh chỉnh siêu tham số bằng `GridSearchCV` trên `StratifiedKFold(5)` và chống rò rỉ dữ liệu (_Data Leakage_) bằng `Pipeline` độc lập cho từng mô hình.
   - Đường cong học tập (_Learning Curves_) phân tích hiện tượng Overfitting/Underfitting.
   - Tối ưu hóa ngưỡng quyết định (_Threshold Tuning_) theo bài toán y tế: Giữ nguyên xác suất hậu nghiệm chuẩn để chọn ngưỡng $p^*$ đạt $\text{Recall} \ge 0.80$ và tối đa hóa Precision nhằm giảm chi phí $\text{Cost} = FP + 5 \times FN$.
   - Ước lượng khoảng tin cậy 95% bằng Percentile Bootstrap ($B=2000$ lần lấy mẫu hoàn lại, có kiểm đếm và báo cáo các lần lặp thiếu lớp).
4. **Điểm cộng sáng tạo (Bonus ML ngoài chương trình giảng dạy)**:
   - **Kiểm định Đối nghịch (Adversarial Validation)**: Sử dụng XGBoost phân biệt nguồn gốc dữ liệu nhằm định lượng mức độ trôi phân phối (_Data Drift_) và xếp hạng các đặc trưng biến đổi mạnh nhất.
   - **Hiệu chuẩn xác suất hậu nghiệm (Post-hoc Probability Calibration)**: Áp dụng Platt Scaling và Isotonic Regression để khắc phục sai lệch tỷ lệ nền khi chuyển giao mô hình xuyên miền (_Cross-domain Transfer_).
   - **Giải thích mô hình cục bộ & toàn cục với SHAP (SHapley Additive exPlanations)**: Bóc tách cơ chế ra quyết định của mô hình cây và truy nguyên các trường hợp dự đoán sai âm tính (_False Negatives_).
   - **Dự đoán Phù hợp Đảm bảo Độ phủ (Conformal Prediction via MAPIE)**: Cung cấp tập dự đoán có bảo chứng thống kê $\{0\}, \{1\}$ hoặc $\{0, 1\}$ (chuyển hội chẩn bác sĩ khi bất định).


In [ ]:
# [Setup & Reproducibility]
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Sklearn imports
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    learning_curve,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Optional scikit-learn 1.6+ FrozenEstimator support
try:
    from sklearn.frozen import FrozenEstimator
except ImportError:
    FrozenEstimator = None

# XGBoost, SHAP & MAPIE
import mapie
import shap
import xgboost as xgb
from xgboost import XGBClassifier

try:
    from mapie.classification import SplitConformalClassifier
except ImportError:
    from mapie.classification import MapieClassifier as SplitConformalClassifier

# Global configuration
warnings.filterwarnings("ignore")
np.random.seed(42)
plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
plt.rcParams.update(
    {
        "font.sans-serif": "DejaVu Sans",
        "axes.edgecolor": "#cccccc",
        "axes.linewidth": 0.8,
    }
)

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)


def save_fig(fig, filename, dpi=200):
    path = os.path.join(FIG_DIR, filename)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f" Saved figure: {path} (DPI={dpi})")


print(
    f"Environment ready | Python {sys.version.split()[0]} | "
    f"sklearn {sklearn.__version__} | xgboost {xgb.__version__} | "
    f"shap {shap.__version__} | mapie {mapie.__version__}"
)


## 2. Tiền xử lý & Căn chỉnh lược đồ (Schema Harmonization)

Hai tập dữ liệu xuất phát từ các nguồn điều tra dịch tễ học và lâm sàng khác nhau với các đơn vị và quy cách đo lường khác biệt:

1. **Lọc dữ liệu ngoại lai sinh học bất khả thi (Cardio Dataset)**:
   - Huyết áp tâm thu (`ap_hi`): $60 \le \text{ap\_hi} \le 250$ mmHg.
   - Huyết áp tâm trương (`ap_lo`): $30 \le \text{ap\_lo} \le 160$ mmHg.
   - Điều kiện sinh lý bắt buộc: $\text{ap\_hi} > \text{ap\_lo}$.
   - Chiều cao (`height`): $120 \le \text{height} \le 220$ cm; Cân nặng (`weight`): $30 \le \text{weight} \le 200$ kg.
2. **Căn chỉnh về 8 đặc trưng lâm sàng dùng chung theo 3 nhóm chuẩn không gian metric**:
   - **Nhóm biến số liên tục (`NUM_FEATURES`)** — điền khuyết `median` + chuẩn hóa `StandardScaler`:
     - `age`: Tuổi bệnh nhân (quy đổi thống nhất về số năm thực).
     - `bmi`: Chỉ số khối cơ thể $\text{BMI} = \frac{\text{weight (kg)}}{(\text{height (m)})^2}$.
     - `sys_bp`: Huyết áp tâm thu (mmHg).
     - `dia_bp`: Huyết áp tâm trương (mmHg).
   - **Nhóm biến thứ bậc (`ORD_FEATURES`)** — điền khuyết `most_frequent` + chuẩn hóa `StandardScaler` để đồng nhất khoảng cách Euclidean (KNN) và phạt điều chuẩn $L_2$ (Logistic Regression, SVM):
     - `chol_level`: Mức Cholesterol theo chuẩn AHA/NCEP ATP III ($1$: Bình thường $<200$, $2$: Ranh giới cao $200\text{–}239$, $3$: Cao $\ge 240$ mg/dL).
     - `gluc_level`: Mức Glucose máu lúc đói theo chuẩn ADA ($1$: Bình thường $<100$, $2$: Tiền đái tháo đường $100\text{–}125$, $3$: Đái tháo đường $\ge 126$ mg/dL).
   - **Nhóm biến nhị phân (`BIN_FEATURES`)** — điền khuyết `most_frequent`, giữ nguyên mã hóa $\{0, 1\}$:
     - `is_male`: Giới tính nhị phân ($1 = \text{Nam}, 0 = \text{Nữ}$).
     - `smoker`: Trạng thái hút thuốc ($1 = \text{Có}, 0 = \text{Không}$).
3. **Phân chia 60% Train / 20% Validation / 20% Test** phân tầng (_Stratified_). Việc điền giá trị thiếu (_Imputation_) và chuẩn hóa (_Standardization_) được đóng gói hoàn toàn trong `Pipeline` độc lập cho từng mô hình trên tập Train để loại bỏ 100% rủi ro rò rỉ dữ liệu (_Data Leakage_).


In [ ]:
# [Data Loading & Biological Filtering]
df_framingham_raw = pd.read_csv("data/raw/framingham.csv")
df_cardio_raw = pd.read_csv("data/raw/cardio_train.csv", sep=";")

print(f"Raw Framingham shape: {df_framingham_raw.shape}")
print(f"Raw Cardio shape:     {df_cardio_raw.shape}")

# Biological sanity filtering for Cardio dataset
c = df_cardio_raw
valid_cardio = (
    c["ap_hi"].between(60, 250)
    & c["ap_lo"].between(30, 160)
    & (c["ap_hi"] > c["ap_lo"])
    & c["height"].between(120, 220)
    & c["weight"].between(30, 200)
)
df_cardio_filtered = df_cardio_raw[valid_cardio].copy()
removed_rows = len(df_cardio_raw) - len(df_cardio_filtered)
print(
    f"Cardio filtered: Removed {removed_rows} invalid physiological rows ({removed_rows / len(df_cardio_raw) * 100:.2f}%). Remaining: {len(df_cardio_filtered)}"
)


In [ ]:
# [Harmonization Function & Unified 3-Group Feature Schema]
NUM_FEATURES = ["age", "bmi", "sys_bp", "dia_bp"]
ORD_FEATURES = ["chol_level", "gluc_level"]
BIN_FEATURES = ["is_male", "smoker"]
COMMON_FEATURES = NUM_FEATURES + ORD_FEATURES + BIN_FEATURES


def harmonize_framingham(df):
    clean = pd.DataFrame(
        {
            "age": df["age"].astype(float),
            "bmi": df["BMI"].astype(float),
            "sys_bp": df["sysBP"].astype(float),
            "dia_bp": df["diaBP"].astype(float),
            "is_male": df["male"].astype(int),
            "smoker": df["currentSmoker"].astype(int),
            "chol_level": np.select(
                [df["totChol"].isna(), df["totChol"] < 200, df["totChol"] < 240],
                [np.nan, 1.0, 2.0],
                default=3.0,
            ),
            "gluc_level": np.select(
                [df["glucose"].isna(), df["glucose"] < 100, df["glucose"] < 126],
                [np.nan, 1.0, 2.0],
                default=3.0,
            ),
            "target": df["TenYearCHD"].astype(int),
        },
        index=df.index,
    )
    return clean[COMMON_FEATURES + ["target"]]


def harmonize_cardio(df):
    clean = pd.DataFrame(
        {
            "age": (df["age"] / 365.25).round(1),
            "bmi": (df["weight"] / ((df["height"] / 100.0) ** 2)).round(2),
            "sys_bp": df["ap_hi"].astype(float),
            "dia_bp": df["ap_lo"].astype(float),
            "is_male": (df["gender"] == 2).astype(int),
            "smoker": df["smoke"].astype(int),
            "chol_level": df["cholesterol"].astype(float),
            "gluc_level": df["gluc"].astype(float),
            "target": df["cardio"].astype(int),
        },
        index=df.index,
    )
    return clean[COMMON_FEATURES + ["target"]]


df_A = harmonize_framingham(df_framingham_raw)
df_B = harmonize_cardio(df_cardio_filtered)

print("--- HARMONIZED DATASET OVERVIEW ---")
print(
    f"Dataset A (Framingham): {df_A.shape[0]} rows | Target Rate: {df_A['target'].mean() * 100:.2f}% (Positives: {df_A['target'].sum()})"
)
print(
    f"Dataset B (Cardio):     {df_B.shape[0]} rows | Target Rate: {df_B['target'].mean() * 100:.2f}% (Positives: {df_B['target'].sum()})"
)
print(f"Numeric Features ({len(NUM_FEATURES)}): {NUM_FEATURES}")
print(f"Ordinal Features ({len(ORD_FEATURES)}): {ORD_FEATURES}")
print(f"Binary Features  ({len(BIN_FEATURES)}): {BIN_FEATURES}")


In [ ]:
# [Exploratory Data Analysis & Missingness Audit]
missing_audit = pd.DataFrame(
    {
        "Missing_A_Count": df_A[COMMON_FEATURES].isna().sum(),
        "Missing_A_Pct(%)": (df_A[COMMON_FEATURES].isna().mean() * 100).round(2),
        "Missing_B_Count": df_B[COMMON_FEATURES].isna().sum(),
        "Missing_B_Pct(%)": (df_B[COMMON_FEATURES].isna().mean() * 100).round(2),
    }
)
print("--- MISSING VALUE AUDIT ---")
display(missing_audit)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, col in zip(axes.flatten(), COMMON_FEATURES):
    if col in NUM_FEATURES:
        for df, lbl, colr in [
            (df_A, "Dataset A (Framingham)", "#1f77b4"),
            (df_B, "Dataset B (Cardio)", "#ff7f0e"),
        ]:
            sns.kdeplot(
                data=df[col].dropna(),
                ax=ax,
                label=lbl,
                color=colr,
                fill=True,
                alpha=0.3,
                lw=2,
            )
        ax.set_title(f"Distribution: {col}", fontsize=13, fontweight="bold")
    else:
        cnt_A = df_A[col].value_counts(normalize=True).sort_index()
        cnt_B = df_B[col].value_counts(normalize=True).sort_index()
        cats = sorted(list(set(cnt_A.index) | set(cnt_B.index)))
        x, bar_w = np.arange(len(cats)), 0.35
        ax.bar(
            x - bar_w / 2,
            [cnt_A.get(c, 0) for c in cats],
            bar_w,
            label="Dataset A",
            color="#1f77b4",
            alpha=0.85,
        )
        ax.bar(
            x + bar_w / 2,
            [cnt_B.get(c, 0) for c in cats],
            bar_w,
            label="Dataset B",
            color="#ff7f0e",
            alpha=0.85,
        )
        ax.set_xticks(x)
        ax.set_xticklabels([str(int(c)) if not pd.isna(c) else "NA" for c in cats])
        ax.set_title(f"Proportion: {col}", fontsize=13, fontweight="bold")
    ax.legend(loc="upper right", frameon=True)

plt.tight_layout()
save_fig(fig, "fig1_eda_distributions.png")
plt.show()

# Correlation Heatmap for Both Datasets
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
for ax, df, title, cmap in [
    (ax1, df_A, "Correlation Heatmap: Dataset A (Framingham)", "Blues"),
    (ax2, df_B, "Correlation Heatmap: Dataset B (Cardio)", "Oranges"),
]:
    sns.heatmap(
        df[COMMON_FEATURES + ["target"]].corr(),
        annot=True,
        fmt=".2f",
        cmap=cmap,
        ax=ax,
        cbar=True,
        square=True,
    )
    ax.set_title(title, fontsize=14, fontweight="bold")

plt.tight_layout()
save_fig(fig, "fig2_correlation_matrix.png")
plt.show()


In [ ]:
# [Train / Validation / Test Splitting: 60 / 20 / 20 Stratified | Independent Unit: 1 Patient per Row]
def split_data(df, target_col="target", random_state=42):
    X, y = df[COMMON_FEATURES].copy(), df[target_col].copy()
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y, test_size=0.40, stratify=y, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


X_train_A, X_val_A, X_test_A, y_train_A, y_val_A, y_test_A = split_data(df_A)
X_train_B, X_val_B, X_test_B, y_train_B, y_val_B, y_test_B = split_data(df_B)

for name, (X_tr, y_tr, X_v, y_v, X_te, y_te) in [
    ("Dataset A", (X_train_A, y_train_A, X_val_A, y_val_A, X_test_A, y_test_A)),
    ("Dataset B", (X_train_B, y_train_B, X_val_B, y_val_B, X_test_B, y_test_B)),
]:
    print(
        f"{name} Splits -> Train: {len(X_tr)} ({y_tr.mean() * 100:.1f}% pos) | Val: {len(X_v)} ({y_v.mean() * 100:.1f}% pos) | Test: {len(X_te)} ({y_te.mean() * 100:.1f}% pos)"
    )


def make_preprocessor():
    # 3-branch ColumnTransformer (num, ord, bin) per pipeline
    imp_scale = lambda strat: Pipeline(
        [("imputer", SimpleImputer(strategy=strat)), ("scaler", StandardScaler())]
    )
    return ColumnTransformer(
        transformers=[
            ("num", imp_scale("median"), NUM_FEATURES),
            ("ord", imp_scale("most_frequent"), ORD_FEATURES),
            (
                "bin",
                Pipeline([("imputer", SimpleImputer(strategy="most_frequent"))]),
                BIN_FEATURES,
            ),
        ],
        verbose_feature_names_out=False,
    )


print(
    "Preprocessing factory make_preprocessor() initialized (3-branch metric-safe & state-isolated)."
)


## 3. Huấn luyện & Đánh giá In-Domain (Mô hình trên lớp)

Trong phần này, chúng ta huấn luyện và so sánh toàn diện 7 thuật toán học máy kinh điển được giảng dạy trong học phần:

1. **Dummy Baseline**: Chiến lược `most_frequent`.
2. **Logistic Regression**: Điều chuẩn L2, tinh chỉnh `C` bằng `GridSearchCV` trên 5-fold CV, giữ nguyên xác suất hậu nghiệm chuẩn để phục vụ tối ưu ngưỡng quyết định y tế và trích xuất Odds Ratio ($e^{w_j}$).
3. **K-Nearest Neighbors (KNN)**: Tinh chỉnh `n_neighbors`, `weights` (`uniform` / `distance`) và bậc khoảng cách Minkowski `p` (`1`, `2`) bằng 5-fold CV trong `Pipeline` có `StandardScaler`.
4. **Gaussian Naive Bayes**: Ước lượng xác suất phân phối Gauss với giả thiết độc lập có điều kiện (đánh giá `CV mean ± SD` trên 5-fold).
5. **Random Forest**: Tập hợp 100 cây quyết định, tinh chỉnh `max_depth` và `min_samples_leaf` bằng 5-fold CV.
6. **XGBoost Classifier**: Gradient Boosting tinh chỉnh `max_depth` và `learning_rate` bằng 5-fold CV, kết hợp **Early Stopping** (`early_stopping_rounds=30` trên lát cắt 10% tách nội bộ từ `X_train` để chọn số cây tối ưu mà không chạm `X_val`), giữ phân phối xác suất gốc trung thực.
7. **Support Vector Machine (SVM)**:
   - Với Dataset A (4.2k mẫu): `SVC(kernel='rbf', probability=True, class_weight='balanced')`, tinh chỉnh `C` và `gamma` qua 5-fold CV (dùng `class_weight='balanced'` để tối ưu biên phân tách cho lớp thiểu số 15.2%, sau đó Platt scaling nội bộ 5-fold ánh xạ ngược điểm quyết định về tỷ lệ nền thực nên xác suất hậu nghiệm và $p^*$ vẫn đồng nhất với Logistic Regression và XGBoost).
   - Với Dataset B (70k mẫu): Huấn luyện Kernel SVM thông thường có độ phức tạp $O(N^2)$ đến $O(N^3)$, rất tốn thời gian. Ta áp dụng xấp xỉ hạt nhân **Nystroem Kernel Approximation** kết hợp **SGDClassifier(loss='log_loss')** (thay cho `loss='hinge'` của Linear SVM chuẩn để xuất trực tiếp xác suất hậu nghiệm phục vụ tối ưu ngưỡng $p^*$ và kiểm định Brier Score), đồng thời tinh chỉnh `gamma` và `alpha` bằng 5-fold CV.

### 3.1. Tiêu chí Đánh giá Lâm sàng (Medical Assessment Metrics):

- Trong sàng lọc y tế, **Âm tính giả (False Negative - FN)** là hiểm họa chết người (bệnh nhân mắc bệnh nhưng bị chẩn đoán nhầm là khỏe mạnh và không được điều trị). Do đó, chi phí sai lầm được định nghĩa:
  $$\text{Cost} = 1 \times FP + 5 \times FN$$
- **Tối ưu ngưỡng quyết định (_Threshold Tuning_)**: Tìm ngưỡng $p^*$ trên tập Validation thỏa mãn điều kiện y tế $\text{Recall} \ge 0.80$ và tối đa hóa Precision (nếu Precision bằng nhau, chọn ngưỡng lớn nhất), sau đó giữ cố định $p^*$ để đánh giá trên tập Test.
- **Bootstrap 95% Confidence Interval**: Đánh giá tính vững của chỉ số ROC-AUC qua $B=2000$ lần lấy mẫu hoàn lại (Percentile Bootstrap), có ghi nhận và báo cáo số lần lấy mẫu thiếu lớp.
- **Learning Curves**: Khảo sát động thái lỗi huấn luyện và kiểm định theo kích thước dữ liệu.


In [ ]:
# [Model Definitions & Hyperparameter Search Spaces]
def get_model_zoo(dataset_name="A"):
    models = {
        "Dummy_Baseline": DummyClassifier(strategy="most_frequent"),
        "Logistic_Regression": LogisticRegression(
            penalty="l2", max_iter=1000, random_state=42
        ),
        "KNN": KNeighborsClassifier(),
        "Gaussian_NB": GaussianNB(),
        "Random_Forest": RandomForestClassifier(
            n_estimators=100, random_state=42, n_jobs=-1
        ),
        "XGBoost": XGBClassifier(
            n_estimators=100, eval_metric="logloss", random_state=42, n_jobs=-1
        ),
    }
    param_grids = {
        "Dummy_Baseline": {},
        "Logistic_Regression": {"classifier__C": [0.01, 0.1, 1.0, 10.0]},
        "KNN": (
            {
                "classifier__n_neighbors": [5, 11, 15, 25],
                "classifier__weights": ["uniform", "distance"],
                "classifier__p": [1, 2],
            }
            if dataset_name == "A"
            else {
                "classifier__n_neighbors": [11, 15, 31],
                "classifier__weights": ["uniform", "distance"],
                "classifier__p": [2],
            }
        ),
        "Gaussian_NB": {},
        "Random_Forest": {
            "classifier__max_depth": [4, 6, 8, 12],
            "classifier__min_samples_leaf": [1, 5],
        },
        "XGBoost": {
            "classifier__max_depth": [2, 3, 4],
            "classifier__learning_rate": [0.03, 0.05, 0.1],
        },
    }

    if dataset_name == "A":
        models["SVM"] = SVC(
            kernel="rbf", probability=True, class_weight="balanced", random_state=42
        )
        param_grids["SVM"] = {
            "classifier__C": [0.1, 1.0, 10.0],
            "classifier__gamma": ["scale", 0.01, 0.1],
        }
    else:
        linear_clf = SGDClassifier(loss="log_loss", penalty="l2", random_state=42)
        models["SVM"] = Pipeline(
            [
                ("nystroem", Nystroem(kernel="rbf", n_components=100, random_state=42)),
                ("clf", linear_clf),
            ]
        )
        param_grids["SVM"] = {
            "classifier__nystroem__gamma": ["scale", 0.1],
            "classifier__clf__alpha": [1e-4, 1e-3, 1e-2],
        }

    return models, param_grids


models_A, grids_A = get_model_zoo("A")
models_B, grids_B = get_model_zoo("B")

print(
    f"Initialized {len(models_A)} models + CV grids for Dataset A and {len(models_B)} models + CV grids for Dataset B."
)


In [ ]:
# [Evaluation, CV Model Selection & Threshold Optimization Engine]
def compute_medical_cost(y_true, y_pred, cost_fp=1.0, cost_fn=5.0):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return fp * cost_fp + fn * cost_fn


def find_best_threshold(y_val, y_val_proba, min_recall=0.80):
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_proba)
    precisions, recalls = precisions[:-1], recalls[:-1]

    valid_mask = recalls >= min_recall
    if np.any(valid_mask):
        valid_prec = precisions[valid_mask]
        max_prec = np.max(valid_prec)
        best_threshold = np.max(thresholds[valid_mask][valid_prec == max_prec])
    else:
        P = float(np.sum(y_val))
        tp = recalls * P
        fn = P - tp
        fp = np.where(precisions > 0, tp * (1.0 / precisions - 1.0), 0.0)
        best_threshold = thresholds[np.argmin(fp + 5.0 * fn)]
    return float(np.clip(best_threshold, 0.05, 0.95))


def bootstrap_roc_auc_ci(
    y_true, y_proba, n_bootstraps=2000, alpha=0.05, random_state=42
):
    rng = np.random.RandomState(random_state)
    bootstrapped_scores, failed_reps = [], 0
    y_true_arr, y_proba_arr = np.array(y_true), np.array(y_proba)
    n = len(y_true_arr)

    for _ in range(n_bootstraps):
        idx = rng.randint(0, n, n)
        if len(np.unique(y_true_arr[idx])) < 2:
            failed_reps += 1
            continue
        bootstrapped_scores.append(roc_auc_score(y_true_arr[idx], y_proba_arr[idx]))

    sorted_scores = np.sort(bootstrapped_scores)
    return (
        float(np.percentile(sorted_scores, (alpha / 2.0) * 100)),
        float(np.percentile(sorted_scores, (1.0 - alpha / 2.0) * 100)),
        failed_reps,
    )


def select_xgb_n_estimators_early_stopping(
    preproc_factory, best_clf_params, X_train, y_train
):
    X_tr_es, X_val_es, y_tr_es, y_val_es = train_test_split(
        X_train, y_train, test_size=0.10, stratify=y_train, random_state=42
    )
    prep = preproc_factory()
    xgb_es = XGBClassifier(
        n_estimators=1000,
        max_depth=best_clf_params.get("classifier__max_depth", 4),
        learning_rate=best_clf_params.get("classifier__learning_rate", 0.05),
        eval_metric="logloss",
        early_stopping_rounds=30,
        random_state=42,
        n_jobs=-1,
    )
    xgb_es.fit(
        prep.fit_transform(X_tr_es),
        y_tr_es,
        eval_set=[(prep.transform(X_val_es), y_val_es)],
        verbose=False,
    )
    return int(xgb_es.best_iteration + 1)


def evaluate_pipeline(
    models_dict,
    param_grids_dict,
    preproc_factory,
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    dataset_name="Dataset",
):
    trained_pipelines, test_probas, optimal_thresholds = {}, {}, {}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []

    print(
        f"\n{'=' * 30} CV MODEL SELECTION & TEST EVALUATION ON {dataset_name} {'=' * 30}"
    )
    for name, clf in models_dict.items():
        base_pipe = Pipeline(
            [("preprocessor", preproc_factory()), ("classifier", clone(clf))]
        )
        grid = param_grids_dict.get(name, {})

        if grid:
            search = GridSearchCV(
                base_pipe, grid, cv=cv, scoring="roc_auc", n_jobs=-1, refit=True
            )
            search.fit(X_train, y_train)
            best_idx = search.best_index_
            cv_mean = float(search.cv_results_["mean_test_score"][best_idx])
            cv_std = float(search.cv_results_["std_test_score"][best_idx])
            best_params, pipe = search.best_params_, search.best_estimator_

            if name == "XGBoost":
                best_n_trees = select_xgb_n_estimators_early_stopping(
                    preproc_factory, best_params, X_train, y_train
                )
                pipe = Pipeline(
                    [
                        ("preprocessor", preproc_factory()),
                        (
                            "classifier",
                            clone(clf).set_params(
                                n_estimators=best_n_trees,
                                max_depth=best_params["classifier__max_depth"],
                                learning_rate=best_params["classifier__learning_rate"],
                            ),
                        ),
                    ]
                )
                es_scores = cross_val_score(
                    pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1
                )
                cv_mean, cv_std = float(np.mean(es_scores)), float(np.std(es_scores))
                pipe.fit(X_train, y_train)
                best_params = {
                    **best_params,
                    "classifier__n_estimators(ES)": best_n_trees,
                }
        else:
            cv_scores = cross_val_score(
                base_pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1
            )
            cv_mean, cv_std = float(np.mean(cv_scores)), float(np.std(cv_scores))
            best_params, pipe = {"default": True}, base_pipe.fit(X_train, y_train)

        trained_pipelines[name] = pipe
        val_proba = pipe.predict_proba(X_val)[:, 1]
        test_proba = pipe.predict_proba(X_test)[:, 1]
        test_probas[name] = test_proba

        best_thresh = (
            0.5
            if name == "Dummy_Baseline"
            else find_best_threshold(y_val, val_proba, min_recall=0.80)
        )
        optimal_thresholds[name] = best_thresh

        test_pred_def = (test_proba >= 0.50).astype(int)
        test_pred_opt = (test_proba >= best_thresh).astype(int)

        if len(np.unique(test_proba)) > 1:
            auc = roc_auc_score(y_test, test_proba)
            ci_low, ci_high, failed_reps = bootstrap_roc_auc_ci(
                y_test, test_proba, n_bootstraps=2000
            )
        else:
            auc, ci_low, ci_high, failed_reps = 0.5, 0.5, 0.5, 0

        ap = average_precision_score(y_test, test_proba)
        rec_opt = recall_score(y_test, test_pred_opt)
        prec_opt = precision_score(y_test, test_pred_opt, zero_division=0)
        f1_opt = f1_score(y_test, test_pred_opt, zero_division=0)
        cost_opt = compute_medical_cost(y_test, test_pred_opt)
        cost_def = compute_medical_cost(y_test, test_pred_def)
        brier = brier_score_loss(y_test, test_proba)

        clean_params = ", ".join(
            f"{k.replace('classifier__', '')}={v}" for k, v in best_params.items()
        )
        results.append(
            {
                "Model": name,
                "Best_Params": clean_params,
                "CV_ROC_AUC": f"{cv_mean:.3f} +/- {cv_std:.3f}",
                "ROC_AUC": auc,
                "AUC_95_CI": f"[{ci_low:.3f}, {ci_high:.3f}]",
                "Boot_Fail": failed_reps,
                "PR_AUC": ap,
                "Threshold_Opt": best_thresh,
                "Recall_Opt": rec_opt,
                "Precision_Opt": prec_opt,
                "F1_Opt": f1_opt,
                "Cost_Default(0.5)": cost_def,
                "Cost_Tuned(p*)": cost_opt,
                "Cost_Reduction": f"{((cost_def - cost_opt) / max(cost_def, 1)) * 100:.1f}%",
                "Brier_Score": brier,
            }
        )
        print(
            f"Finished {name:20s} | CV: {cv_mean:.3f}+/-{cv_std:.3f} | Test AUC: {auc:.3f} (fail={failed_reps}) | "
            f"PR-AUC: {ap:.3f} | p*: {best_thresh:.3f} | Recall@p*: {rec_opt:.3f} | Cost: {cost_def:.0f} -> {cost_opt:.0f}"
        )

    return pd.DataFrame(results), trained_pipelines, test_probas, optimal_thresholds


df_results_A, pipes_A, probas_A, thresh_A = evaluate_pipeline(
    models_A,
    grids_A,
    make_preprocessor,
    X_train_A,
    y_train_A,
    X_val_A,
    y_val_A,
    X_test_A,
    y_test_A,
    "Dataset A (Framingham)",
)

df_results_B, pipes_B, probas_B, thresh_B = evaluate_pipeline(
    models_B,
    grids_B,
    make_preprocessor,
    X_train_B,
    y_train_B,
    X_val_B,
    y_val_B,
    X_test_B,
    y_test_B,
    "Dataset B (Cardio)",
)


In [ ]:
# [In-Domain ROC and PR Curves Visualization]
def plot_curves(y_test, probas_dict, dataset_label, filename):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    colors = [
        "#7f7f7f",
        "#1f77b4",
        "#2ca02c",
        "#d62728",
        "#9467bd",
        "#8c564b",
        "#e377c2",
    ]

    for (name, proba), color in zip(probas_dict.items(), colors):
        if name == "Dummy_Baseline":
            continue
        fpr, tpr, _ = roc_curve(y_test, proba)
        auc = roc_auc_score(y_test, proba)
        ax1.plot(fpr, tpr, lw=2, color=color, label=f"{name} (AUC = {auc:.3f})")

        prec, rec, _ = precision_recall_curve(y_test, proba)
        ap = average_precision_score(y_test, proba)
        ax2.plot(rec, prec, lw=2, color=color, label=f"{name} (PR-AUC = {ap:.3f})")

    ax1.plot([0, 1], [0, 1], "k--", lw=1.5, alpha=0.7)
    ax1.set(
        title=f"ROC Curves - {dataset_label}",
        xlabel="False Positive Rate (FPR)",
        ylabel="True Positive Rate (TPR / Recall)",
    )
    ax1.legend(loc="lower right", frameon=True)

    ax2.axhline(
        y=y_test.mean(),
        color="k",
        linestyle="--",
        lw=1.5,
        alpha=0.7,
        label=f"No-skill Baseline ({y_test.mean():.2f})",
    )
    ax2.set(
        title=f"Precision-Recall Curves - {dataset_label}",
        xlabel="Recall",
        ylabel="Precision",
    )
    ax2.legend(loc="upper right", frameon=True)

    plt.tight_layout()
    save_fig(fig, filename)
    plt.show()


plot_curves(
    y_test_A, probas_A, "Dataset A (Framingham)", "fig3_roc_pr_curves_framingham.png"
)
plot_curves(y_test_B, probas_B, "Dataset B (Cardio)", "fig4_roc_pr_curves_cardio.png")


In [ ]:
# [Learning Curves Analysis: Stratified 5-Fold with +/- 1 SD]
cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_sizes_grid = np.linspace(0.2, 1.0, 5)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
models_lc = [
    (
        "Logistic_Regression",
        "Learning Curves: Logistic Regression (Dataset A)",
        [0.65, 0.85],
    ),
    ("XGBoost", "Learning Curves: XGBoost (Dataset A)", [0.60, 1.00]),
]

for ax, (m_name, title, ylim) in zip(axes, models_lc):
    sizes, tr_scores, val_scores = learning_curve(
        pipes_A[m_name],
        X_train_A,
        y_train_A,
        cv=cv_strat,
        scoring="roc_auc",
        train_sizes=train_sizes_grid,
        n_jobs=-1,
    )
    tr_m, tr_s = np.mean(tr_scores, axis=1), np.std(tr_scores, axis=1)
    val_m, val_s = np.mean(val_scores, axis=1), np.std(val_scores, axis=1)

    ax.plot(sizes, tr_m, "o-", color="#1f77b4", lw=2, label="Training ROC-AUC")
    ax.fill_between(sizes, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color="#1f77b4")
    ax.plot(
        sizes, val_m, "s--", color="#ff7f0e", lw=2, label="5-Fold CV ROC-AUC (+/- 1 SD)"
    )
    ax.fill_between(sizes, val_m - val_s, val_m + val_s, alpha=0.15, color="#ff7f0e")

    ax.set(
        title=title, xlabel="Training Sample Size", ylabel="ROC-AUC Score", ylim=ylim
    )
    ax.legend(loc="lower right", frameon=True)

plt.tight_layout()
save_fig(fig, "fig5_learning_curves.png")
plt.show()


In [ ]:
# [In-Domain Results Summary Table & Comparison Plot]
print("--- SUMMARY TABLE: DATASET A (FRAMINGHAM) ---")
display(df_results_A.sort_values(by="ROC_AUC", ascending=False))

print("\n--- SUMMARY TABLE: DATASET B (CARDIO) ---")
display(df_results_B.sort_values(by="ROC_AUC", ascending=False))

# Comparison Bar Chart
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
models_plot = df_results_A[df_results_A["Model"] != "Dummy_Baseline"]["Model"].values
x, bar_w = np.arange(len(models_plot)), 0.35

chart_configs = [
    (
        axes[0],
        "ROC_AUC",
        "ROC-AUC Score",
        [0.5, 0.88],
        "In-Domain ROC-AUC Comparison Across Models",
    ),
    (
        axes[1],
        "F1_Opt",
        "F1 Score at Optimal Medical Threshold (p*)",
        [0.0, 0.88],
        "In-Domain F1 Score at Medical Threshold (p*)",
    ),
]

for ax, metric, ylabel, ylim, title in chart_configs:
    v_A = [
        df_results_A[df_results_A["Model"] == m][metric].values[0] for m in models_plot
    ]
    v_B = [
        df_results_B[df_results_B["Model"] == m][metric].values[0] for m in models_plot
    ]
    ax.bar(
        x - bar_w / 2,
        v_A,
        bar_w,
        label="Dataset A (Framingham)",
        color="#1f77b4",
        alpha=0.85,
    )
    ax.bar(
        x + bar_w / 2,
        v_B,
        bar_w,
        label="Dataset B (Cardio)",
        color="#ff7f0e",
        alpha=0.85,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(models_plot, rotation=25, ha="right", fontsize=11)
    ax.set(ylabel=ylabel, ylim=ylim, title=title)
    ax.legend(loc="upper left", frameon=True)

plt.tight_layout()
save_fig(fig, "fig6_in_domain_comparison.png")
plt.show()


# Logistic Regression Odds Ratios (exp(w_j)): Conditional Association (Association != Causation)
def extract_odds_ratios(pipe_lr):
    prep, clf = pipe_lr.named_steps["preprocessor"], pipe_lr.named_steps["classifier"]
    scales = {
        f: float(s)
        for b in ["num", "ord"]
        for f, s in zip(
            prep.transformers_[0 if b == "num" else 1][2],
            prep.named_transformers_[b].named_steps["scaler"].scale_,
        )
    }
    return pd.DataFrame(
        [
            {
                "Feature": f,
                "Coef": round(float(w), 4),
                "OR_1SD": round(float(np.exp(w)), 4),
                "OR_Unit": round(float(np.exp(w / scales.get(f, 1.0))), 4),
            }
            for f, w in zip(prep.get_feature_names_out(), clf.coef_[0])
        ]
    )


df_odds_cmp = extract_odds_ratios(pipes_A["Logistic_Regression"]).merge(
    extract_odds_ratios(pipes_B["Logistic_Regression"]),
    on="Feature",
    suffixes=("_A", "_B"),
)
print(
    "\n--- LOGISTIC REGRESSION ODDS RATIOS (exp(w_j)) | NOTE: Association != Causation ---"
)
display(df_odds_cmp)


## 4. Thực nghiệm Cross-Domain, Phân tích Lỗi & Khắc phục Trôi dữ liệu

Khi mô hình đạt hiệu năng cao trên tập nội bộ (_In-Domain_), các kỹ sư thường ngộ nhận mô hình đã sẵn sàng triển khai. Phần thực nghiệm nâng cao này chứng minh hiện tượng suy giảm hiệu năng nghiêm trọng khi chuyển giao xuyên miền và áp dụng 4 kỹ thuật Học Máy hiện đại để kiểm định và giải quyết:

### 4.1. Đánh giá Zero-shot Cross-Domain ($A \to B$ và $B \to A$)

- Ta đem mô hình huấn luyện trên Tập A (Framingham, tỷ lệ bệnh ~15%) kiểm thử trực tiếp trên Tập B (Cardio, tỷ lệ bệnh ~50%) và ngược lại.
- **Hiện tượng**: Brier Score tăng vọt, sai lệch dự báo trầm trọng do:
  1. _Covariate Shift_: $P_A(X) \ne P_B(X)$ (Khác biệt về độ tuổi, huyết áp giữa hai quần thể).
  2. _Base-rate / Prior Shift_: $P_A(Y=1) \approx 0.15$ trong khi $P_B(Y=1) \approx 0.50$. Ngưỡng phân loại $p^*$ được tối ưu trên tập nguồn trở nên hoàn toàn sai lệch khi áp dụng sang tập đích!
  3. _Concept Shift (Target Definition Shift)_: $P_A(Y \mid X) \ne P_B(Y \mid X)$. Hai tập có bản chất nhãn lâm sàng khác nhau: Tập A dự báo biến cố mạch vành trong 10 năm tới (_TenYearCHD_ — bài toán tiên lượng dài hạn), trong khi Tập B chẩn đoán hiện diện bệnh tim mạch tại thời điểm khám (_cardio_ — bài toán chẩn đoán hiện thời). Do đó, cùng một hồ sơ đặc trưng $X$, xác suất mắc bệnh tức thời và xác suất phát bệnh sau 10 năm không tương đương nhau, làm thay đổi cả quy luật ánh xạ giữa $X$ và $Y$.

---

### 4.2. Kiểm định Đối nghịch (Adversarial Validation)

- **Bản chất Học máy**: Gộp toàn bộ dữ liệu $X_A$ và $X_B$, gán nhãn miền: $Y_{\text{domain}} = 1$ nếu mẫu thuộc Tập B, và $0$ nếu thuộc Tập A.
- Huấn luyện mô hình `XGBClassifier` để phân loại nhãn miền này.
- **Ý nghĩa định lượng**:
  - Nếu hai tập dữ liệu đồng nhất (không có trôi dạt): ROC-AUC của mô hình sẽ tiệm cận $0.50$ (mô hình không thể đoán được mẫu đến từ đâu).
  - Nếu ROC-AUC $\gg 0.70$: Chứng minh hai tập dữ liệu trôi phân phối nghiêm trọng.
  - **Feature Importance của bộ phân loại miền**: Chỉ điểm chính xác đặc trưng nào trong lâm sàng bị trôi lệch mạnh nhất giữa hai quốc gia/quần thể.

---

### 4.3. Hiệu chuẩn Xác suất Hậu nghiệm Hai chiều (Bidirectional Post-hoc Probability Calibration)

- Áp dụng `CalibratedClassifierCV` (Platt Scaling và Isotonic Regression) trên lát cắt Validation nhỏ của miền đích cho **cả hai chiều chuyển giao ($A \to B$ và $B \to A$)**.
- Khắc phục đồng thời hiện tượng dự báo quá thấp (_Under-confidence_ khi chuyển $15\% \to 50\%$) lẫn dự báo quá cao (_Over-confidence_ khi chuyển $50\% \to 15\%$) mà không cần huấn luyện lại mô hình nguồn.
- Vẽ đường cong tin cậy đối xứng (_Reliability Diagrams_) và định lượng mức giảm sai số qua Brier Score.

---

### 4.4. Khám phá Cơ chế Ra Quyết định & Phân tích Âm tính giả với SHAP

- Sử dụng `shap.TreeExplainer` cho mô hình XGBoost trên cả hai tập dữ liệu.
- So sánh xếp hạng tầm quan trọng đặc trưng toàn cục (_Global Feature Importance_) giữa 2 miền: Yếu tố nào quyết định nguy cơ tim mạch ở Mỹ (Tập A) so với Đông Âu (Tập B)?
- **Phân tích Cục bộ Trường hợp Bỏ sót Bệnh (False Negative Case Study)**: Trực quan hóa thác giá trị SHAP (_Waterfall Plot_) trên các bệnh nhân thực tế có biến cố tim mạch nhưng bị mô hình dự đoán xác suất thấp, tìm ra nguyên nhân lâm sàng dẫn đến nhận định sai lầm của mô hình.

---

### 4.5. Dự đoán Phù hợp Đảm bảo Độ phủ (Conformal Prediction via MAPIE)

- Trong chẩn đoán y tế, một dự đoán nhị phân cứng $(0$ hoặc $1)$ có thể gây nguy hiểm khi mô hình nằm trong vùng biên không chắc chắn (_Epistemic Uncertainty_).
- Áp dụng **Conformal Prediction** thông qua thư viện `MAPIE`, thiết lập mức ý nghĩa thống kê $\alpha = 0.10$ để đạt bảo chứng toán học:
  $$P(Y \in C(X)) \ge 1 - \alpha = 90\%$$
- Đầu ra của mô hình là một tập dự đoán:
  - $\{0\}$: Chắc chắn khỏe mạnh.
  - $\{1\}$: Chắc chắn nguy cơ cao (cần can thiệp).
  - $\{0, 1\}$: **Bất định (Uncertain)** $\to$ Tự động gắn cờ chuyển hồ sơ bệnh nhân cho Bác sĩ chuyên khoa hội chẩn trực tiếp!


In [ ]:
# [4.1. Zero-shot Cross-Domain Transfer Evaluation]
def eval_cross_domain(
    model_name, pipe_src, X_tgt, y_tgt, src_name, tgt_name, thresh_src
):
    tgt_proba = pipe_src.predict_proba(X_tgt)[:, 1]
    pred_def = (tgt_proba >= 0.50).astype(int)
    pred_thresh = (tgt_proba >= thresh_src).astype(int)

    return {
        "Transfer": f"{src_name} -> {tgt_name}",
        "Model": model_name,
        "Cross_ROC_AUC": roc_auc_score(y_tgt, tgt_proba),
        "Cross_PR_AUC": average_precision_score(y_tgt, tgt_proba),
        "Recall@0.5": recall_score(y_tgt, pred_def),
        "Recall@Src_Thresh": recall_score(y_tgt, pred_thresh),
        "Cost@0.5": compute_medical_cost(y_tgt, pred_def),
        "Cost@Src_Thresh": compute_medical_cost(y_tgt, pred_thresh),
        "Brier_Score": brier_score_loss(y_tgt, tgt_proba),
    }


cross_results = []
for m in ["Logistic_Regression", "Random_Forest", "XGBoost"]:
    cross_results.append(
        eval_cross_domain(
            m, pipes_A[m], X_test_B, y_test_B, "Dataset A", "Dataset B", thresh_A[m]
        )
    )
    cross_results.append(
        eval_cross_domain(
            m, pipes_B[m], X_test_A, y_test_A, "Dataset B", "Dataset A", thresh_B[m]
        )
    )

df_cross = pd.DataFrame(cross_results)
print("--- ZERO-SHOT CROSS-DOMAIN EVALUATION RESULTS ---")
display(df_cross)

print(
    "Note (B -> A Cost@Src_Thresh > Cost@0.5): Balanced Cardio thresholds (thresh_B ~ 0.38-0.39) "
    "are too low for imbalanced Framingham (15.2% pos) under over-confident B -> A probabilities, "
    "inflating False Positives and raising total Cost."
)


In [ ]:
# [4.2. Adversarial Validation: Quantifying Data Drift (Leakage-Free)]
# Prepare balanced combined dataset with domain target (0 = Dataset A, 1 = Dataset B)
X_adv = pd.concat(
    [X_train_A, X_train_B.sample(n=len(X_train_A), random_state=42)],
    ignore_index=True,
)
y_domain = np.repeat([0, 1], len(X_train_A))

# Split BEFORE preprocessing to prevent data leakage into test fold
X_adv_tr, X_adv_te, y_adv_tr, y_adv_te = train_test_split(
    X_adv, y_domain, test_size=0.33, stratify=y_domain, random_state=42
)

adv_pipe = Pipeline(
    [
        ("preprocessor", make_preprocessor()),
        (
            "classifier",
            XGBClassifier(
                n_estimators=100,
                max_depth=3,
                learning_rate=0.05,
                eval_metric="logloss",
                random_state=42,
            ),
        ),
    ]
)
adv_pipe.fit(X_adv_tr, y_adv_tr)

adv_probs = adv_pipe.predict_proba(X_adv_te)[:, 1]
adv_auc = roc_auc_score(y_adv_te, adv_probs)

print(
    f"\n{'*' * 50}\nADVERSARIAL VALIDATION ROC-AUC: {adv_auc:.4f}\n"
    + (
        "CONCLUSION: Severe distribution shift detected between Dataset A (Framingham) and Dataset B (Cardio)!"
        if adv_auc > 0.70
        else "CONCLUSION: Distributions are relatively homogenous."
    )
    + f"\n{'*' * 50}\n"
)

feat_names = adv_pipe.named_steps["preprocessor"].get_feature_names_out().tolist()
importances = adv_pipe.named_steps["classifier"].feature_importances_
indices = np.argsort(importances)[::-1]

print("Top Drift Features (Gain Importance):")
for rank, idx in enumerate(indices, 1):
    print(f"  {rank}. {feat_names[idx]:12s}: {importances[idx]:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(feat_names)), importances[indices[::-1]], color="#d62728", alpha=0.85)
ax.set_yticks(range(len(feat_names)))
ax.set_yticklabels([feat_names[i] for i in indices[::-1]], fontsize=11)
ax.set(
    xlabel="Adversarial Feature Importance (Gain)",
    title=f"Adversarial Validation: Features Driving Population Drift (AUC = {adv_auc:.3f})",
)

plt.tight_layout()
save_fig(fig, "fig7_adversarial_validation_drift.png")
plt.show()


In [ ]:
# [4.3. Bidirectional Post-hoc Probability Calibration: A -> B and B -> A]
def fit_calibrators(pipe_src, X_val_tgt, y_val_tgt):
    est_kw = (
        {"estimator": FrozenEstimator(pipe_src)}
        if FrozenEstimator is not None
        else {"estimator": pipe_src, "cv": "prefit"}
    )
    c_iso = CalibratedClassifierCV(**est_kw, method="isotonic").fit(
        X_val_tgt, y_val_tgt
    )
    c_sig = CalibratedClassifierCV(**est_kw, method="sigmoid").fit(X_val_tgt, y_val_tgt)
    return c_iso, c_sig


def eval_cal_row(setting, y_true, proba, threshold=0.50):
    pred = (proba >= threshold).astype(int)
    return {
        "Setting": setting,
        "Threshold": threshold,
        "Brier_Score": brier_score_loss(y_true, proba),
        "Recall": recall_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "Cost": compute_medical_cost(y_true, pred),
    }


cal_data, cal_clinical_rows = {}, []
print("--- BIDIRECTIONAL BRIER SCORE BEFORE & AFTER CALIBRATION ---")

directions = [
    ("A -> B", pipes_A["XGBoost"], X_val_B, y_val_B, X_test_B, y_test_B),
    ("B -> A", pipes_B["XGBoost"], X_val_A, y_val_A, X_test_A, y_test_A),
]

for dir_lbl, pipe_src, X_v, y_v, X_te, y_te in directions:
    c_iso, c_sig = fit_calibrators(pipe_src, X_v, y_v)
    probs = {
        "Raw XGBoost": pipe_src.predict_proba(X_te)[:, 1],
        "Sigmoid / Platt": c_sig.predict_proba(X_te)[:, 1],
        "Isotonic": c_iso.predict_proba(X_te)[:, 1],
    }
    cal_data[dir_lbl] = probs

    b_raw, b_sig, b_iso = (
        brier_score_loss(y_te, probs["Raw XGBoost"]),
        brier_score_loss(y_te, probs["Sigmoid / Platt"]),
        brier_score_loss(y_te, probs["Isotonic"]),
    )
    print(
        f"Transfer {dir_lbl} | Raw: {b_raw:.4f} | "
        f"Sigmoid: {b_sig:.4f} (-{(b_raw - b_sig) / b_raw * 100:.1f}%) | "
        f"Isotonic: {b_iso:.4f} (-{(b_raw - b_iso) / b_raw * 100:.1f}%)"
    )

    for name, p in [
        ("Raw", probs["Raw XGBoost"]),
        ("Sigmoid", probs["Sigmoid / Platt"]),
        ("Isotonic", probs["Isotonic"]),
    ]:
        cal_clinical_rows.append(
            eval_cal_row(f"{dir_lbl} | {name} @ 0.50", y_te, p, 0.50)
        )

    # Split X_val_tgt 50/50 for leakage-free calibration + target threshold optimization (Recall >= 0.80)
    Xv_fit, Xv_th, yv_fit, yv_th = train_test_split(
        X_v, y_v, test_size=0.50, stratify=y_v, random_state=42
    )
    c_iso_s, c_sig_s = fit_calibrators(pipe_src, Xv_fit, yv_fit)
    for name, c_est in [("Sigmoid", c_sig_s), ("Isotonic", c_iso_s)]:
        p_star = find_best_threshold(
            yv_th, c_est.predict_proba(Xv_th)[:, 1], min_recall=0.80
        )
        cal_clinical_rows.append(
            eval_cal_row(
                f"{dir_lbl} | {name} @ p* (cal)",
                y_te,
                c_est.predict_proba(X_te)[:, 1],
                p_star,
            )
        )

df_cal_clinical = pd.DataFrame(cal_clinical_rows)
print("\n--- CLINICAL SCREENING METRICS BEFORE & AFTER CALIBRATION (XGBoost) ---")
display(df_cal_clinical)

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
curve_configs = [
    (
        axes[0],
        y_test_B,
        cal_data["A -> B"],
        "Cross-Domain A -> B (Framingham -> Cardio)",
        10,
        "uniform",
    ),
    (
        axes[1],
        y_test_A,
        cal_data["B -> A"],
        "Cross-Domain B -> A (Cardio -> Framingham, Quantile Bins)",
        8,
        "quantile",
    ),
]

for ax, y_te, p_dict, title, n_bins, strat in curve_configs:
    for (lbl, p), col in zip(p_dict.items(), ["#d62728", "#1f77b4", "#2ca02c"]):
        prob_true, prob_pred = calibration_curve(y_te, p, n_bins=n_bins, strategy=strat)
        ax.plot(
            prob_pred,
            prob_true,
            marker="o",
            lw=2,
            color=col,
            label=f"{lbl} (Brier = {brier_score_loss(y_te, p):.3f})",
        )
    ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration", alpha=0.7)
    ax.set(
        title=title,
        xlabel="Mean Predicted Probability",
        ylabel="Empirical True Fraction of Positives",
    )
    ax.legend(loc="upper left", frameon=True)

plt.tight_layout()
save_fig(fig, "fig8_calibration_curves.png")
plt.show()


In [ ]:
# [4.4. Multi-Domain SHAP Global Comparison (US vs Eastern Europe) & False Negative Case Study]
def extract_shap(pipe, X_data):
    preproc = pipe.named_steps["preprocessor"]
    X_trans = preproc.transform(X_data)
    exp = shap.TreeExplainer(pipe.named_steps["classifier"])(X_trans)
    cols = preproc.get_feature_names_out().tolist()
    return X_trans, exp, cols, dict(zip(cols, np.mean(np.abs(exp.values), axis=0)))


X_test_A_trans, shap_explanation_A, shap_cols_A, mean_shap_A = extract_shap(
    pipes_A["XGBoost"], X_test_A
)
_, shap_explanation_B, _, mean_shap_B = extract_shap(
    pipes_B["XGBoost"], X_test_B.sample(n=2000, random_state=42)
)

df_shap_cmp = pd.DataFrame(
    {
        "Feature": shap_cols_A,
        "Dataset A (Framingham - US)": [mean_shap_A[f] for f in shap_cols_A],
        "Dataset B (Cardio - Eastern Europe)": [mean_shap_B[f] for f in shap_cols_A],
    }
).sort_values(by="Dataset A (Framingham - US)", ascending=True)

print("--- MULTI-DOMAIN MEAN |SHAP| FEATURE IMPORTANCE ---")
display(df_shap_cmp.sort_values(by="Dataset A (Framingham - US)", ascending=False))

# 2-Panel Plot: Left = Cross-population Comparison, Right = Dataset A Beeswarm
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6.5))
y_idx, bar_h = np.arange(len(df_shap_cmp)), 0.38
ax1.barh(
    y_idx - bar_h / 2,
    df_shap_cmp["Dataset A (Framingham - US)"],
    bar_h,
    label="Dataset A: Framingham (US)",
    color="#1f77b4",
    alpha=0.85,
)
ax1.barh(
    y_idx + bar_h / 2,
    df_shap_cmp["Dataset B (Cardio - Eastern Europe)"],
    bar_h,
    label="Dataset B: Cardio (Eastern Europe)",
    color="#ff7f0e",
    alpha=0.85,
)
ax1.set_yticks(y_idx)
ax1.set_yticklabels(df_shap_cmp["Feature"], fontsize=11)
ax1.set(
    xlabel="Mean |SHAP Value| (Average Impact Magnitude)",
    title="Multi-Domain SHAP Comparison: US vs Eastern Europe",
)
ax1.legend(loc="lower right", frameon=True)

plt.sca(ax2)
shap.summary_plot(
    shap_explanation_A.values,
    X_test_A_trans,
    feature_names=shap_cols_A,
    show=False,
    plot_size=None,
)
ax2.set_title(
    "SHAP Value Polarity (Dataset A - Framingham)", fontsize=13, fontweight="bold"
)

plt.tight_layout()
save_fig(fig, "fig9_shap_global_summary.png")
plt.show()

# False Negative Case Analysis
y_pred_A_opt = (probas_A["XGBoost"] >= thresh_A["XGBoost"]).astype(int)
fn_indices = np.where((y_test_A.values == 1) & (y_pred_A_opt == 0))[0]
print(
    f"Total False Negatives on Test A: {len(fn_indices)} patients out of {y_test_A.sum()} actual positive cases."
)

if len(fn_indices) > 0:
    worst_fn_idx = fn_indices[np.argmin(probas_A["XGBoost"][fn_indices])]
    p_raw = X_test_A.iloc[worst_fn_idx]
    print(
        f"Analyzing Worst False Negative Patient (Index #{worst_fn_idx}): Pred={probas_A['XGBoost'][worst_fn_idx]:.3f} vs True=1\n"
        f"Patient Profile: Age={p_raw['age']}, BP={p_raw['sys_bp']}/{p_raw['dia_bp']}, "
        f"BMI={p_raw['bmi']}, Male={int(p_raw['is_male'])}, "
        f"Smoker={int(p_raw['smoker'])}, Chol={int(p_raw['chol_level'])}, Gluc={int(p_raw['gluc_level'])}"
    )

    fig = plt.figure(figsize=(10, 6))
    shap_sample = shap_explanation_A[worst_fn_idx]
    shap_sample.feature_names = shap_cols_A
    shap.plots.waterfall(shap_sample, show=False)
    plt.title(
        f"SHAP Waterfall: Why Model Missed Patient #{worst_fn_idx} (False Negative)",
        fontsize=13,
        fontweight="bold",
        pad=15,
    )
    plt.tight_layout()
    save_fig(plt.gcf(), "fig10_shap_local_false_negatives.png")
    plt.show()


In [ ]:
# [4.5. Conformal Prediction via MAPIE: Trustworthy Screening (Dataset A & Dataset B)]
ALPHA = 0.10  # 90% guaranteed coverage


def run_conformal_eval(pipe_model, X_cal, y_cal, X_test, y_test, dataset_label):
    try:
        mapie_clf = SplitConformalClassifier(
            estimator=pipe_model, confidence_level=1.0 - ALPHA, prefit=True
        )
        mapie_clf.conformalize(X_cal, y_cal)
        _, y_ps = mapie_clf.predict_set(X_test)
    except Exception:
        mapie_clf = SplitConformalClassifier(
            estimator=pipe_model, method="lac", cv="prefit"
        )
        mapie_clf.fit(X_cal, y_cal)
        _, y_ps = mapie_clf.predict(X_test, alpha=ALPHA)

    emp_cov = float(np.mean([y_ps[i, y_test.iloc[i], 0] for i in range(len(y_test))]))
    p0 = float(np.mean((y_ps[:, 0, 0] == 1) & (y_ps[:, 1, 0] == 0)))
    p1 = float(np.mean((y_ps[:, 0, 0] == 0) & (y_ps[:, 1, 0] == 1)))
    p01 = float(np.mean((y_ps[:, 0, 0] == 1) & (y_ps[:, 1, 0] == 1)))
    p_empty = float(np.mean(y_ps[:, :, 0].sum(axis=1) == 0))

    print(
        f"\n--- CONFORMAL PREDICTION (MAPIE) ON {dataset_label} ---\n"
        f"Target Coverage (1 - alpha): {1 - ALPHA:.1%} | Empirical Coverage: {emp_cov:.1%}\n"
        f"  - Low Risk Only {{0}}:    {p0 * 100:6.2f}% (Safe to discharge)\n"
        f"  - High Risk Only {{1}}:   {p1 * 100:6.2f}% (Definite intervention)\n"
        f"  - Uncertain {{0, 1}}:     {p01 * 100:6.2f}% (FLAG FOR DOCTOR CONSULTATION)\n"
        f"  - Empty Sets {{}}:        {p_empty * 100:6.2f}%"
    )
    return emp_cov, [p0 * 100, p1 * 100, p01 * 100]


cov_A, pcts_A = run_conformal_eval(
    pipes_A["Random_Forest"],
    X_val_A,
    y_val_A,
    X_test_A,
    y_test_A,
    "Dataset A (Framingham)",
)
cov_B, pcts_B = run_conformal_eval(
    pipes_B["Random_Forest"], X_val_B, y_val_B, X_test_B, y_test_B, "Dataset B (Cardio)"
)

for m in ["Logistic_Regression", "XGBoost"]:
    run_conformal_eval(
        pipes_A[m], X_val_A, y_val_A, X_test_A, y_test_A, f"Dataset A ({m})"
    )

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
cats = ["Low Risk\n{0}", "High Risk\n{1}", "Uncertain {0, 1}\n(Doctor Referral)"]
configs = [
    (axes[0], pcts_A, cov_A, "Dataset A: Framingham (Imbalanced 15.2%)"),
    (axes[1], pcts_B, cov_B, "Dataset B: Cardio (Balanced 49.5%)"),
]

for ax, pcts, cov, title in configs:
    bars = ax.bar(
        cats, pcts, color=["#2ca02c", "#d62728", "#ff7f0e"], alpha=0.85, width=0.5
    )
    ax.set(
        ylabel="Percentage of Test Population (%)",
        ylim=[0, 100],
        title=f"{title}\n(Empirical Coverage: {cov * 100:.1f}% >= 90%)",
    )
    for bar in bars:
        yval = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            yval + 2,
            f"{yval:.1f}%",
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold",
        )

plt.tight_layout()
save_fig(fig, "fig11_conformal_prediction_coverage.png")
plt.show()


## 5. Kết luận, Bàn luận & Khuyến nghị Triển khai

---

### 1. Tổng kết Kết quả Nghiên cứu Thực nghiệm

1. **Hiệu năng Mô hình In-Domain (Nội miền) & Tinh chỉnh CV**:
   - **Thiết kế kiểm định chuẩn mực**: Đơn vị quan sát độc lập là từng bệnh nhân riêng biệt (mỗi dòng là một cá nhân độc lập, hợp lệ cho phân tầng `StratifiedKFold(5)` và chia 60/20/20). Toàn bộ cấu hình siêu tham số được chọn qua `GridSearchCV` trên Train (`CV mean ± SD`), `XGBoost` dùng thêm Early Stopping (`early_stopping_rounds=30` trên lát cắt 10% nội bộ từ Train) và đánh giá trên Test bằng Percentile Bootstrap ($B=2000$, `Boot_Fail=0` trên mọi mô hình).
   - Trên Tập A (Framingham, mất cân bằng 15.19%): **SVM RBF** (`C=1.0, gamma=0.01`, CV `0.722 ± 0.017`, Test ROC-AUC **0.747** [0.698, 0.794], PR-AUC **0.388**, Brier **0.114**) và **Logistic Regression** (`C=0.1`, CV `0.720 ± 0.021`, Test ROC-AUC **0.746** [0.698, 0.792], PR-AUC **0.399**, Brier **0.113**) dẫn đầu, theo sau là **Random Forest** (`max_depth=8, min_samples_leaf=5`, **0.739**) và **XGBoost** (`max_depth=2, learning_rate=0.05, n_estimators=181`, **0.733**).
   - Trên Tập B (Cardio, 68.619 mẫu sạch): **XGBoost** (`max_depth=4, learning_rate=0.05, n_estimators=120`, CV `0.801 ± 0.006`, Test ROC-AUC **0.798** [0.790, 0.806]) và **Random Forest** (`max_depth=8, min_samples_leaf=5`, CV `0.800 ± 0.006`, Test **0.797** [0.790, 0.805]) dẫn đầu, trong khi **Nystroem + SGD** (`loss='log_loss'`, `gamma=0.1, alpha=1e-4`, Test **0.795**) giải quyết triệt để nút thắt $O(N^3)$ của Kernel SVM mà vẫn xuất xác suất chuẩn; tinh chỉnh `KNN` lên `k=31` nâng Test ROC-AUC lên **0.787** và giảm Cost **30.1%**.
   - **Tối ưu hóa ngưỡng quyết định y tế ($p^*$)**: Giữ nguyên xác suất hậu nghiệm chuẩn rồi tối ưu ngưỡng theo $\text{Recall} \ge 0.80$ giúp giảm **4.6% đến 19.8%** tổng chi phí y tế trên Tập A và **22.0% đến 34.8%** trên Tập B.
   - **Diễn giải Odds Ratio (`LogisticRegression`, $e^{w_j}$, Association $\ne$ Causation)**: Trên Tập A, `age` (`OR/1SD = 1.7324`), `is_male` (`OR = 1.5327`), `sys_bp` (`OR/1SD = 1.4071`) và `smoker` (`OR = 1.3993`) làm tăng mạnh khả năng mắc biến cố 10 năm; trên Tập B, `sys_bp` (`OR/1SD = 2.6420`), `chol_level` (`OR/1SD = 1.4012`, `OR/Unit = 1.6422`) và `age` (`OR/1SD = 1.3993`) chi phối rõ nét.

2. **Bài học Sống còn về Đánh giá Xuyên miền (Cross-Domain Shift)**:
   - Khi đưa mô hình Tập A dự báo trực tiếp sang Tập B ($A \to B$) tại ngưỡng mặc định $p=0.50$, Recall sụt thảm hại xuống chỉ còn **0.29% – 0.93%** (`0.93%` ở XGBoost) và Brier Score vọt lên **0.344 – 0.356**. Ngược lại ở chiều $B \to A$, việc áp thẳng ngưỡng tối ưu của Tập B (`0.379`–`0.393`) lên Tập A (tỷ lệ bệnh 15.19%) làm tổng chi phí tăng so với ngưỡng mặc định $0.50$ (**510 $\to$ 544** với LR, **569 $\to$ 573** với RF, **551 $\to$ 579** với XGBoost) do bùng nổ Dương tính giả.
   - **Adversarial Validation** (tách tập kiểm tra trước khi fit preprocessor để chống rò rỉ) đạt **ROC-AUC = 0.9841**, chỉ điểm các đặc trưng trôi phân phối mạnh nhất giữa Mỹ và Đông Âu: `chol_level` (0.4763), `smoker` (0.1143), `dia_bp` (0.1014) và `gluc_level` (0.0963).
   - **Hiệu chuẩn Xác suất Hậu nghiệm Hai chiều (Platt / Isotonic)** phục hồi chuẩn xác suất cho cả hai hướng trôi lệch:
     - Chiều $A \to B$ (khắc phục _Under-confidence_): Hạ Brier Score từ **0.3460 xuống 0.2138** (**giảm 38.2%** với Isotonic).
     - Chiều $B \to A$ (khắc phục _Over-confidence_): Hạ Brier Score từ **0.3310 xuống 0.1235** (**giảm 62.7%** với Sigmoid/Platt).
     - **Tác động lâm sàng sau hiệu chuẩn (`XGBoost`)**: Trên chiều $A \to B$, chỉ hiệu chuẩn Isotonic đã nâng Recall@0.50 từ **0.93% lên 71.39%** và hạ Cost từ **33.650 xuống 12.409 (-63.1%)**; khi kết hợp hiệu chuẩn với tái tối ưu ngưỡng trên lát cắt đích ($p^* = 0.351$), Recall đạt **85.95%** và Cost giảm còn **8.931 (-73.5%)**. Ở chiều $B \to A$, nếu chỉ hiệu chuẩn rồi giữ ngưỡng $0.50$, Recall tụt về **0.00%–1.55%** do tỷ lệ nền thật chỉ 15.19%; bắt buộc phải kết hợp Sigmoid calibration với tái tối ưu ngưỡng đích ($p^* = 0.093$) để khôi phục Recall lên **86.82%** (Cost **584**, Brier **0.1235**).
   - **Giới hạn do Dịch chuyển Khái niệm (Concept Shift)**: Hiệu chuẩn hậu nghiệm (Platt / Isotonic) chỉ xử lý được sai lệch tỷ lệ nền (_Prior Shift_) và độ lệch điểm cắt xác suất, không thể triệt tiêu hoàn toàn khác biệt quy luật $P(Y \mid X)$ do Tập A tiên lượng nguy cơ 10 năm (`TenYearCHD`) còn Tập B chẩn đoán bệnh hiện tại (`cardio`). Điều này giải thích trực tiếp kết quả SHAP đa miền và Odds Ratio: `age` chi phối mạnh nhất ở Tập A do tích lũy nguy cơ theo thời gian, còn `sys_bp` và `chol_level` áp đảo ở Tập B vì phản ánh bệnh lý đang hiện hữu.

3. **Tính Minh bạch & Đảm bảo An toàn Y tế (Trustworthy AI)**:
   - **SHAP Đa miền (Mỹ vs Đông Âu)**: Trên Tập A (dự báo biến cố 10 năm), `age` (`0.536`) và `sys_bp` (`0.300`) chi phối mạnh nhất; trong khi trên Tập B (chẩn đoán hiện diện bệnh), `sys_bp` (`0.833`) áp đảo và `chol_level` (`0.202`) có sức nặng cao gấp ~15 lần so với Tập A (`0.013`).
   - **SHAP Waterfall ca Âm tính giả #672**: Giải mã ca bệnh 35 tuổi, nam, hút thuốc, huyết áp 112/62.5 mmHg bị đánh giá nguy cơ thấp ($p=0.022$) do tuổi trẻ kéo mạnh giá trị SHAP về miền âm, lấn át yếu tố hút thuốc và giới tính nam.
   - **Conformal Prediction (MAPIE, $\alpha = 0.10$)**: Đảm bảo độ phủ thực nghiệm vượt cam kết **90%** (**90.8%** trên Framingham và **90.1%** trên Cardio với `Random_Forest`). Trên Tập A lệch lớp, hệ thống khoanh vùng **16.39%** ca bất định $\{0, 1\}$ cho bác sĩ hội chẩn, loại trừ an toàn **83.61%** ca $\{0\}$ và cho **0.00%** ca $\{1\}$ (xu hướng đồng nhất khi kiểm chứng thêm trên `Logistic_Regression`: độ phủ **92.0%**, `{0, 1}` **19.58%**, `{1}` **0.00%** và `XGBoost`: độ phủ **91.2%**, `{0, 1}` **16.63%**, `{1}` **0.12%** — phản ánh bản chất dữ liệu thiểu số 15.19% chưa đủ chắc chắn để loại bỏ hoàn toàn nhãn $0$ ở mức tin cậy 90%); trên Tập B cân bằng, phân tách rõ **27.43%** $\{0\}$, **28.80%** $\{1\}$ và **43.78%** $\{0, 1\}$.

---

### 2. Sơ đồ Hình ảnh Đã Xuất vào Thư mục `figures/`

Mọi đồ thị phục vụ bài thuyết trình đã được lưu với chuẩn độ phân giải cao (200 DPI):

- `fig1_eda_distributions.png`: Phân phối so sánh 8 đặc trưng lâm sàng giữa 2 tập dữ liệu.
- `fig2_correlation_matrix.png`: Ma trận tương quan đặc trưng của từng tập.
- `fig3_roc_pr_curves_framingham.png`: Đường cong ROC & PR trên tập Framingham.
- `fig4_roc_pr_curves_cardio.png`: Đường cong ROC & PR trên tập Cardio.
- `fig5_learning_curves.png`: Đường cong học tập Stratified 5-Fold kèm dải độ lệch chuẩn $\pm 1\sigma$.
- `fig6_in_domain_comparison.png`: Biểu đồ cột so sánh ROC-AUC & F1-Score của 7 mô hình.
- `fig7_adversarial_validation_drift.png`: Xếp hạng đặc trưng trôi dữ liệu bằng Kiểm định Đối nghịch.
- `fig8_calibration_curves.png`: Đường cong tin cậy hiệu chuẩn xác suất hậu nghiệm hai chiều ($A \to B$ & $B \to A$).
- `fig9_shap_global_summary.png`: Đối sánh SHAP toàn cục Mỹ vs Đông Âu và phân bố cực tính Tập A.
- `fig10_shap_local_false_negatives.png`: Phân tích ca bệnh Âm tính giả bằng SHAP Waterfall.
- `fig11_conformal_prediction_coverage.png`: Tỷ lệ phân bổ các tập dự đoán tin cậy qua MAPIE trên cả 2 tập dữ liệu.
